## 1. Instalasi dan Persiapan Model

Pertama, kita instal Unsloth versi terbaru dan memuat model **Llama-3.2-1B-Instruct**. Model dimuat dalam mode **4-bit** untuk menghemat memori VRAM GPU.

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git@nightly git+https://github.com/unslothai/unsloth-zoo.git

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


## 2. Konfigurasi LoRA Adapters

Kita pasang adapter LoRA pada model. Dengan ini, kita hanya akan melatih sekitar 1% hingga 10% dari total parameter model, sehingga proses *fine-tuning* menjadi sangat cepat dan ringan.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.3.4 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


<a name="Data"></a>
### 3. Persiapan & Format Dataset

Karena dataset kita sudah menggunakan format percakapan standar **Hugging Face**:

```
{"role": "...", "content": "..."}
```

Kita bisa langsung menerapkan *chat template* bawaan Llama-3.1 tanpa perlu melakukan konversi format.

In [ ]:
from unsloth.chat_templates import get_chat_template

# 1. Siapkan Template Chat (Llama-3.1 kompatibel dengan 3.2)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

# 2. Fungsi untuk memetakan kolom "messages" menjadi teks utuh
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

## 4. Memuat Seluruh Dataset

Kita muat sekaligus ketiga *split dataset* (Train, Valid, Test) dan langsung terapkan fungsi format yang sudah dibuat ke semua data.

In [ ]:
from datasets import load_dataset

# ==========================================
# MUAT SEMUA DATASET SEKALIGUS
# ==========================================
data_files = {
    "train": "/content/train.jsonl",
    "validation": "/content/valid.jsonl",
    "test": "/content/test.jsonl"
}

dataset = load_dataset("json", data_files=data_files)

# Terapkan format template chat
dataset = dataset.map(formatting_prompts_func, batched = True)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1408 [00:00<?, ? examples/s]

Map:   0%|          | 0/171 [00:00<?, ? examples/s]

Map:   0%|          | 0/177 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['messages', 'text'],
        num_rows: 1408
    })
    validation: Dataset({
        features: ['messages', 'text'],
        num_rows: 171
    })
    test: Dataset({
        features: ['messages', 'text'],
        num_rows: 177
    })
})

## 5. Pengecekan Hasil Format

Mari kita intip struktur data setelah diformat.

In [ ]:
dataset['train'][0]

{'messages': [{'role': 'user',
   'content': 'Tolong artikan istilah kodifikasi ini dong.'},
  {'role': 'assistant',
   'content': 'Disusunnya ketentuan-ketentuan hukum dalam sebuah kitab secara sistematik dan teratur.'}],
 'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTolong artikan istilah kodifikasi ini dong.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nDisusunnya ketentuan-ketentuan hukum dalam sebuah kitab secara sistematik dan teratur.<|eot_id|>'}

**[Penting]** Anda akan melihat teks *"Cutting Knowledge Date..."* disisipkan oleh sistem. Ini adalah *System Prompt* otomatis bawaan dari Llama 3.1 Instruct. **Ini sangat normal dan aman dibiarkan**.

In [ ]:
dataset['train'][5]['text']

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nBantu saya memahami istilah anti dumping duty/bea anti dumping. Apa itu?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAturan hukum yang disepakati oleh negara-negara anggota GATT yang menetapkan standar prosedural maupun substantif untuk mencegah tindakan dumping di negara-negara anggota GATT yang menandatangani aturan tersebut.<|eot_id|>'

<a name="Train"></a>
## 6. Konfigurasi SFTTrainer

Kita gunakan `SFTTrainer` dari library `trl`. Kita akan menjalankan iterasi penuh sebanyak **3 Epoch**, dan model akan diuji coba (dievaluasi) menggunakan dataset validasi setiap **20 langkah**.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["validation"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # Biarkan False karena dataset kita berupa tanya-jawab pendek
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3, # Training penuh 3 putaran
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,

        # --- PENGATURAN EVALUASI (VALIDASI) ---
        eval_strategy = "steps",
        eval_steps = 20,
        save_strategy = "steps",
        save_steps = 20,

        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none"
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1408 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/171 [00:00<?, ? examples/s]

## 7. Melatih Khusus pada Jawaban Asisten *(Response Masking)*

Agar model menjadi cerdas, kita instruksikan model untuk **hanya belajar dari respons AI (Assistant)**. Fungsi ini akan mengabaikan kalkulasi *Loss* dari teks instruksi pengguna (User).

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=6):   0%|          | 0/1408 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/1408 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/171 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/171 [00:00<?, ? examples/s]

Mari kita buktikan apakah penyembunyian *(masking)* berhasil dilakukan pada kolom "Labels":

In [ ]:
# Tampilan Input asli
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nBantu saya memahami istilah anti dumping duty/bea anti dumping. Apa itu?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAturan hukum yang disepakati oleh negara-negara anggota GATT yang menetapkan standar prosedural maupun substantif untuk mencegah tindakan dumping di negara-negara anggota GATT yang menandatangani aturan tersebut.<|eot_id|>'

In [ ]:
# Tampilan Labels (Perhatikan bagian pertanyaan user dihilangkan dan hanya menyisakan jawaban)
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                         Aturan hukum yang disepakati oleh negara-negara anggota GATT yang menetapkan standar prosedural maupun substantif untuk mencegah tindakan dumping di negara-negara anggota GATT yang menandatangani aturan tersebut.<|eot_id|>'

> Jika output kedua hanya menampilkan jawaban asisten, berarti masking sukses!





## 8. Mulai Proses Fine-tuning!

Semua sudah siap. Jalankan *cell* di bawah ini untuk memulai proses *training*.

In [ ]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
1.107 GB of memory reserved.


### Train the model

In [ ]:
import logging
import warnings

# Matikan peringatan dari library transformers & unsloth
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("unsloth").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

In [ ]:
trainer_stats = trainer.train()

Step,Training Loss,Validation Loss
20,2.173216,2.016866
40,2.370936,1.999676
60,2.584174,1.976287
80,2.273053,1.951351
100,2.120731,1.967210
120,1.894640,1.924469
140,1.862274,1.940674
160,2.061714,1.934459
180,1.577728,1.916401
200,2.044139,1.920720


In [ ]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

583.6066 seconds used for training.
9.73 minutes used for training.
Peak reserved memory = 2.861 GB.
Peak reserved memory for training = 1.754 GB.
Peak reserved memory % of max memory = 19.646 %.
Peak reserved memory for training % of max memory = 12.044 %.


<a name="Inference"></a>
### 9. Pengujian Model (Inference)

Setelah proses *training* selesai, mari kita uji kecerdasan model ini! Kita gunakan fungsi `FastLanguageModel.for_inference` untuk mengaktifkan mode inferensi bawaan yang **2x lebih cepat**. Kita juga mengatur `temperature` dan `min_p` untuk menjaga keseimbangan antara kreativitas dan akurasi logika model.

In [ ]:
from unsloth.chat_templates import get_chat_template

# Pastikan template chat menggunakan standar Llama 3.1
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) # Aktifkan inferensi cepat

# Siapkan prompt pertanyaan
messages = [
    {"role": "user", "content": "Lanjutkan bilangan fibonnaci ini: 1, 1, 2, 3, 5, 8,"},
]

# Ubah prompt menjadi tensor yang bisa dibaca GPU
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Wajib agar model tahu ini gilirannya menjawab
    return_tensors = "pt",
).to("cuda")

# Hasilkan jawaban
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 64,
    use_cache = True,
    temperature = 0.7,
    min_p = 0.1
)
print(tokenizer.batch_decode(outputs))

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nLanjutkan bilangan fibonnaci ini: 1, 1, 2, 3, 5, 8,<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610.<|eot_id|>']

### 10. Pengujian dengan Output Teks Berjalan (Streaming)

Menunggu seluruh paragraf selesai di-*generate* terkadang memakan waktu. Kita bisa menggunakan `TextStreamer` agar jawaban model muncul kata demi kata secara *real-time* ke layar, persis seperti antarmuka ChatGPT.

In [ ]:
FastLanguageModel.for_inference(model)

# Mari kita uji dengan kasus perpajakan lokal
messages = [
    {"role": "user", "content": "Saya punya warung makan kecil di pasar Tenggarong, omset Rp 50 juta/bulan. Berapa persen pajak yang harus dibayar dan pasal mana yang mengatur tarif lengkap beserta cara penghitungannya?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True) # skip_prompt menyembunyikan pertanyaan kita dari hasil cetak

_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    use_cache = True,
    temperature = 0.75,
    min_p = 0.1
)

1. pajak yang harus dibayar: pajak yang terjadi pada barang-barang masuk ke Indonesia, yaitu pajak yang harus dibayar adalah pajak pajak yang mengatur tarif lengkap beserta cara penghitungannya adalah: (1) pajak yang harus dibayar pada barang-barang masuk ke Indonesia, yaitu pajak pajak yang mengatur tarif lengkap beserta cara penghitungannya adalah: (1) pajak pajak yang mengatur tarif lengkap beserta cara penghitungannya adalah: (1) pajak pajak yang mengatur tarif lengkap,


<a name="Save"></a>
### 11. Menyimpan Model (LoRA Adapters)

Tahap ini digunakan untuk menyimpan bobot (*weights*) hasil *fine-tuning*. Perlu diingat, ini **hanya menyimpan LoRA adapters** yang ukurannya sangat kecil, bukan keseluruhan model Llama yang bergiga-giga.

In [ ]:
# 1. Simpan ke sistem lokal (folder Colab)
model.save_pretrained("lex01_lora")
tokenizer.save_pretrained("lex01_lora")

# 2. Simpan secara online (Unggah ke Hugging Face)
model.push_to_hub("nxvay/lex01_lora", token = "")
tokenizer.push_to_hub("nxvay/lex01_lora", token = "")

('lex01_lora/tokenizer_config.json',
 'lex01_lora/chat_template.jinja',
 'lex01_lora/tokenizer.json')

### 12. Konversi ke Format GGUF (Untuk Llama.cpp / Ollama)

Agar model yang sudah dilatih ini bisa Anda jalankan secara *offline* di MacBook lokal Anda (menggunakan Ollama atau LM Studio), kita harus mengekspornya ke format **GGUF**. Unsloth sudah mendukung fitur ini secara *native*.

Pada skrip ini, kita mengekspornya ke dalam format presisi tinggi **16-bit (f16)** dan mengunggahnya ke repo Hugging Face Anda.

In [ ]:
# Simpan versi GGUF ke folder lokal Colab
if True:
    model.save_pretrained_gguf("lex01_lora", tokenizer, quantization_method = "f16")

# Unggah versi GGUF ke Hugging Face Hub
if True:
    model.push_to_hub_gguf(
        "nxvay/lex01_lora",
        tokenizer,
        quantization_method = "f16",
        token = ""
    )

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:04<00:00, 64.90s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:11<00:00, 71.20s/it]


Unsloth: Merge process complete. Saved to `/content/lex01_lora`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['lex01_lora_gguf/llama-3.2-1b-instruct.F16.gguf']
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['lex01_lora_gguf/llama-3.2-1b-instruct.F16.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model lex01_lora_gguf/llama-3.2-1b-instruct.F16.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to lex01_lora_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f lex01_lora_gguf/Modelfile
Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloadin

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:22<00:00, 82.57s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:57<00:00, 57.31s/it]


Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_w4fnil77`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_w4fnil77_gguf/llama-3.2-1b-instruct.F16.gguf']
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/unsloth_gguf_w4fnil77_gguf/llama-3.2-1b-instruct.F16.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /tmp/

In [ ]:
# Ekstra: Unggah model LoRA dalam format 'Merged' yang disatukan
model.push_to_hub_merged(
    "nxvay/lex01_lora",
    tokenizer,
    save_method = "lora",
    token = ""
)

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:18<00:00, 78.30s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]